In [ ]:
# Run 005: chỉ inference, không cập nhật trọng số; đọc controls và reference riêng.
from pathlib import Path
import sys, subprocess, json, zipfile, traceback, importlib.metadata, platform
from IPython.display import display, FileLink

loaded=[name for name in ('peft','torchao','chronos','transformers','quantstream')
        if any(module==name or module.startswith(name+'.') for module in sys.modules)]
if loaded:
    raise RuntimeError('Restart Session rồi Run All: '+', '.join(loaded))
SOURCE_COMMIT='d6bbe46d8452422f9a606704041af3e2ceb8fee8'
WORK=Path('/kaggle/working/quantstream_run005_retry001')
WORK.mkdir(parents=True,exist_ok=True)
CODE=WORK/('source-'+SOURCE_COMMIT[:12])
OUTPUT=WORK/'run-005-inference'
ARCHIVE=WORK.parent/'QuantStream_Run005_Retry001_Output.zip'
if OUTPUT.exists() or ARCHIVE.exists():
    raise FileExistsError('Đã có evidence Run 005; tải lại để review, không xóa nhằm chạy lại.')
if not CODE.exists():
    subprocess.check_call(['git','init','-q',str(CODE)])
    subprocess.check_call(['git','-C',str(CODE),'remote','add','origin',
                           'https://github.com/taiiswibu/quantstream-crypto-research.git'])
    subprocess.check_call(['git','-C',str(CODE),'fetch','-q','--depth','1','origin',SOURCE_COMMIT])
    subprocess.check_call(['git','-C',str(CODE),'checkout','-q','--detach','FETCH_HEAD'])
actual=subprocess.check_output(['git','-C',str(CODE),'rev-parse','HEAD'],text=True).strip()
if actual!=SOURCE_COMMIT or subprocess.check_output(['git','-C',str(CODE),'status','--porcelain'],text=True).strip():
    raise ValueError('Cần source SHA đã pin và checkout sạch')
sys.path.insert(0,str(CODE))
print('Pinned source:',SOURCE_COMMIT)

In [ ]:
# Giữ runtime đã đối chiếu probe; lỗi thì dừng, không bỏ kiểm tra.
import os, shutil
from quantstream.inference_setup import prepare_runtime
os.environ['HF_HUB_DISABLE_XET']='1'
old_lock=json.loads((CODE/'reports/experiments/run-004-seed-stability/kaggle-preflight-lock-rerun-001.json').read_text())
SETUP=WORK/'setup-evidence'
try:
    setup_report=prepare_runtime(old_lock['environment'],SETUP)
except BaseException:
    OUTPUT.mkdir(parents=True,exist_ok=False)
    if SETUP.exists():
        shutil.copytree(SETUP,OUTPUT/'setup',dirs_exist_ok=True)
    (OUTPUT/'setup-error.txt').write_text(traceback.format_exc(),encoding='utf-8')
    (OUTPUT/'status.json').write_text(json.dumps({'status':'setup_failed','optimizer_steps':0}),encoding='utf-8')
    with zipfile.ZipFile(ARCHIVE,'w',zipfile.ZIP_DEFLATED) as archive:
        for path in OUTPUT.rglob('*'):
            if path.is_file():
                archive.write(path,path.relative_to(OUTPUT))
    display(FileLink(str(ARCHIVE)))
    raise
print('Setup completed. NumPy imported:',setup_report['loaded_runtime']['modules']['numpy']['version'])
print('Full legacy inventory matches Run 004. No fitting or backward pass was performed.')

In [ ]:
# Biểu đồ Run 004 đọc log cũ đã xác minh, không phải lần train mới.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

VISUALS=WORK/'visual-review'
VISUALS.mkdir(exist_ok=True)
review=CODE/'reports/experiments/run-004-seed-stability'
loss=pd.read_csv(review/'training-loss-verified.csv')
historical=pd.read_csv(review/'metrics-verified.csv')
if len(loss)!=600 or set(loss.seed)!={42,123,2026}:
    raise ValueError('Run 004 verified training records differ')
fig,axes=plt.subplots(2,2,figsize=(14,9),layout='constrained')
for seed,part in loss.groupby('seed',sort=True):
    part=part.sort_values('step')
    line=axes[0,0].plot(part.step,part.loss,alpha=.15)[0]
    axes[0,0].plot(part.step,part.loss.rolling(20,min_periods=1).mean(),
                   color=line.get_color(),label=f'Seed {seed}: rolling 20')
    axes[0,1].plot(part.step,part.learning_rate,label=f'Seed {seed}')
    axes[1,0].plot(part.step,part.grad_norm,label=f'Seed {seed}',alpha=.7)
axes[0,0].set(title='Run 004 training loss: minibatch / rolling mean',xlabel='Optimizer step',ylabel='Training objective')
axes[0,1].set(title='Recorded learning-rate schedule',xlabel='Optimizer step',ylabel='Learning rate')
axes[1,0].set(title='Recorded gradient norm',xlabel='Optimizer step',ylabel='Gradient norm')
pooled=historical[historical.asset.eq('ALL')].copy()
names=[f'LoRA seed {int(row.seed)}' if row.model=='lora_B' else row.model for row in pooled.itertuples()]
axes[1,1].barh(names,pooled.mae*1e4,color=['#64748b','#94a3b8','#2563eb','#0891b2','#0d9488'])
axes[1,1].invert_yaxis()
axes[1,1].set(title='Same 512 development windows; lower MAE is better',xlabel='MAE (basis points of return)')
for ax in axes.flat:
    ax.grid(alpha=.2)
for ax in (axes[0,0],axes[0,1],axes[1,0]):
    ax.legend(fontsize=8)
fig.suptitle('Run 004 evidence — no step-wise validation curve was recorded',fontsize=14)
fig.savefig(VISUALS/'run004-training-and-baselines.png',dpi=150)
plt.show()
plt.close(fig)
display(historical[['asset','model','seed','rows','mae','mse','directional_accuracy']])
settings={key:old_lock[key] for key in ('model_id','revision','steps','seeds','context_length','batch_size','learning_rate')}
settings.update({key:old_lock['lora'][key] for key in ('r','lora_alpha','lora_dropout','target_modules')})
display(pd.DataFrame({'setting':list(settings),'recorded_value':[str(value) for value in settings.values()]}))

In [ ]:
# Chạy đủ đối chứng; không chọn riêng lượt có điểm tốt.
from quantstream.inference_lab import prepare_inputs,run_inference_lab
from quantstream.seed_lab import archive_evidence
try:
    snapshot,reference=prepare_inputs(Path('/kaggle/input'),WORK)
    report=run_inference_lab(snapshot,reference,OUTPUT,device='cuda')
    print('Optimizer steps:',report['optimizer_steps'])
    print('Controls passed:',report['controls_passed'])
    print('Run 004 reference reproduced:',report['reference_reproduced'])
    print('Đã xong inference diagnostic. Tải QuantStream_Run005_Retry001_Output.zip và gửi lại.')
except BaseException:
    if not OUTPUT.exists():
        OUTPUT.mkdir(parents=True)
        (OUTPUT/'input-error.txt').write_text(traceback.format_exc(),encoding='utf-8')
        (OUTPUT/'status.json').write_text(json.dumps({'status':'input_failed','optimizer_steps':0}),encoding='utf-8')
    raise
finally:
    if OUTPUT.exists() and SETUP.exists():
        shutil.copytree(SETUP,OUTPUT/'setup',dirs_exist_ok=True)
    saved=archive_evidence(OUTPUT,ARCHIVE)
    if saved:
        display(FileLink(saved))

In [ ]:
# Completed khác controls_passed và reference_reproduced.
if report['completed_passes']!=8:
    raise ValueError('Need the completed eight-pass diagnostic before plotting')
reference=pd.read_parquet(OUTPUT/'reference_predictions.parquet')
base=pd.read_parquet(OUTPUT/'base_control.parquet')
passes={row['case']:pd.read_parquet(OUTPUT/'passes'/(row['case']+'.parquet')) for row in report['passes']}
labels=['zero_return','zero_shot',*passes]
fig,axes=plt.subplots(1,3,figsize=(16,6),layout='constrained')
for ax,asset in zip(axes,('ALL','BTCUSDT','ETHUSDT')):
    mask=np.ones(len(reference),dtype=bool) if asset=='ALL' else reference.symbol.eq(asset).to_numpy()
    truth=reference.target_return.to_numpy()[mask]
    values=[np.mean(abs(truth)),np.mean(abs(base.prediction.to_numpy()[mask]-truth))]
    values.extend(np.mean(abs(frame.prediction.to_numpy()[mask]-truth)) for frame in passes.values())
    ax.barh(labels,np.asarray(values)*1e4,color=['#64748b','#94a3b8']+['#2563eb' if name.startswith('eval') else '#d97706' for name in passes])
    ax.invert_yaxis()
    ax.set(title=f'{asset}: {mask.sum()} rows',xlabel='MAE (basis points of return)')
    ax.grid(axis='x',alpha=.2)
fig.suptitle('Run 005: fixed adapter, every predeclared pass — lower error is better')
fig.savefig(VISUALS/'run005-baselines-and-passes.png',dpi=150)
plt.show()
plt.close(fig)
fig,axes=plt.subplots(2,1,figsize=(14,8),layout='constrained')
for ax,asset in zip(axes,('BTCUSDT','ETHUSDT')):
    mask=reference.symbol.eq(asset).to_numpy()
    times=pd.to_datetime(reference.loc[mask,'decision_time'],utc=True)
    ax.scatter(times,reference.target_return.to_numpy()[mask]*1e4,s=10,alpha=.5,label='Actual target return')
    for case,color in (('eval_42_a','#2563eb'),('dropout_42_a','#d97706')):
        ax.scatter(times,passes[case].prediction.to_numpy()[mask]*1e4,s=10,alpha=.6,label=case,color=color)
    ax.axhline(0,color='#64748b',linewidth=.7)
    ax.set(title=f'{asset}: the same sampled development windows',xlabel='Decision time (UTC)',ylabel='Return (basis points)')
    ax.legend(fontsize=8)
    ax.grid(alpha=.2)
fig.savefig(VISUALS/'run005-actual-and-predicted-returns.png',dpi=150)
plt.show()
plt.close(fig)
checks=pd.DataFrame(report['checks']).T
display(checks)
display(pd.DataFrame(report['passes'])[['case','rng_seed','model_training_before','model_training_after',
    'active_dropout_modules','merged_lora_layers','state_unchanged','trainable_parameters','optimizer_steps']])
config=json.loads((OUTPUT/'base_model_config.json').read_text())
keys=('model_type','architectures','d_model','d_ff','d_kv','num_layers','num_heads','dropout_rate','dense_act_fn','feed_forward_proj')
display(pd.DataFrame({'config_key':keys,'actual_value':[str(config.get(key,'not recorded in this config')) for key in keys]}))
import shutil
shutil.copytree(VISUALS,OUTPUT/'figures',dirs_exist_ok=True)
saved=archive_evidence(OUTPUT,ARCHIVE)
display(FileLink(saved))
print('ZIP updated with figures. All model decisions and optimizer_steps=0 remain unchanged.')